# **Setup**

In [ ]:
# Estou utilizando o drive para carregar os datasets direto de lá
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install torch
!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.6.0+cu124.html
!pip install torch-geometric
!pip install torch-geometric-temporal
!pip install numpy pandas tqdm scikit-learn matplotlib loguru torchmetrics timesfm

In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/ColabData/LABIA/DatasetsTSFM')

In [ ]:
# basic
import os
import pickle
import warnings
import numpy as np
import pandas as pd
from tqdm import tqdm
# pre processing
from sklearn import preprocessing as pre
# NN
import torch
import torch.nn as nn
from torch import Tensor
import torch.nn.functional as F
import torch.optim as optim
from torch.nn import MSELoss
from torch_geometric.nn import GCNConv
# val and plot
from torchmetrics.regression import R2Score
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.metrics import mean_absolute_error
from loguru import logger as log
from val import calculate_metrics
# plot
import matplotlib.pyplot as plt
# foundation model
import timesfm
from functools import reduce



In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")

# **Experimento**

In [ ]:
SEED = 1345
def seed_everything(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
seed_everything(SEED)
warnings.filterwarnings('ignore')

In [ ]:
print(torch.__version__)
print(torch.version.cuda)

In [ ]:
sb = pd.read_parquet("/content/drive/MyDrive/ColabData/LABIA/DatasetsTSFM/loader_03-04_2024.parquet")
sb.head()

In [ ]:
sb.shape

In [ ]:
plt.figure(figsize=(20, 5))
plt.plot(sb["125960550"])
plt.show()

In [ ]:
# define X and Y
sbx = sb.query("index <= '2024-03-31 23:59:59'")
sbx.shape, sb.shape

In [ ]:
# define X and Y
sby = sb.query("index > '2024-03-31 23:59:59'")
sby.shape, sb.shape

In [ ]:
pred_len = abs(sbx.shape[0] - sb.shape[0])
pred_len

In [ ]:
sbx

In [ ]:
model_name = 'timesfm-2.0-500m-pytorch'

if model_name == 'timesfm-2.0-500m-pytorch':
  tfm = timesfm.TimesFm(
      hparams=timesfm.TimesFmHparams(
          backend="gpu",
          per_core_batch_size=40,
          horizon_len=pred_len,
          num_layers=50,
          use_positional_embedding=False,
          context_len=2048,

      ),
      checkpoint=timesfm.TimesFmCheckpoint(
                    huggingface_repo_id=(''.join(('google/', model_name))),
      )
)

elif model_name == 'timesfm-1.0-200m-pytorch':
  tfm = timesfm.TimesFm(
      hparams=timesfm.TimesFmHparams(
          backend="gpu",
          per_core_batch_size=40,
          horizon_len=pred_len,
      ),
  checkpoint=timesfm.TimesFmCheckpoint(
                    huggingface_repo_id=(''.join(('google/', model_name))),
      )
  )

In [ ]:
nodes = sb.columns
nodes

In [ ]:
scores_error = {'node':[], 'mae': [], 'mse': [], 'r2': [], 'mape': []}
targets = {}
for node in nodes:
    targets[node] = []
    forecast, experimental_quantile_forecast = tfm.forecast(
            [sbx[node]],
            freq=[0],
    )

    y_hat = torch.tensor(forecast[0])
    y_pred =  y_hat.cpu().data.numpy()
    y_true = sby[node].values

    scores_error['node'].append(node)
    scores_error['mse'].append(mean_squared_error(y_true, y_pred))
    scores_error['mae'].append(mean_absolute_error(y_true, y_pred))
    scores_error['r2'].append(r2_score(y_true, y_pred))
    scores_error['mape'].append(mean_absolute_percentage_error(y_true, y_pred))

    targets[node].append({"input":  np.array(sbx[node]),
                              'true': y_true,
                              'pred': y_pred,
                              'node': node
                             })


In [ ]:
with open(f'{model_name}-targets-long.pkl', 'wb') as f:
    pickle.dump(targets, f)

In [ ]:
df_results = pd.DataFrame(scores_error)
df_results

In [ ]:
df_results["model"] = model_name

In [ ]:
df_results[['model', 'node', 'mae', 'mse', 'r2', 'mape']]

In [ ]:
df_results.to_csv(''.join((model_name, '-long.csv')))